In [1]:
import os
import random
import numpy as np
import torch

def seed_everything(seed: int = 42):
    # Python
    random.seed(seed)
    # Numpy
    np.random.seed(seed)
    # PyTorch (CPU)
    torch.manual_seed(seed)
    # PyTorch (GPU)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinismo (importante!)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Algumas libs usam isso
    os.environ["PYTHONHASHSEED"] = str(seed)
    # Para transformers (às vezes ajuda)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"Seed definida como {seed}")

seed_everything(42)

Seed definida como 42


In [2]:
import numpy as np
import os
from datetime import datetime
import pandas as pd
from tqdm import tqdm

import datasets
import evaluate

import torch
import torch.nn as nn

from transformers import AutoTokenizer, BertConfig, BertModel, BertPreTrainedModel

from sklearn.metrics import f1_score, accuracy_score

/home/guilhermelima/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Mapeamento real do dataset de treino (ordenado por frequência, não alfabeticamente)
DEPREL_LABELS = ['det', 'nsubj', 'root', 'obj', 'xcomp', 'punct', 'mark',
                 'advcl', 'case', 'obl', 'amod', 'conj', 'cc', 'nmod', 'advmod',
                 'flat:name', 'ccomp', 'cop', 'acl', 'nummod', 'acl:relcl',
                 'ccomp:speech', 'parataxis', 'csubj', 'aux:pass', 'appos', 'fixed',
                 'nsubj:pass', 'aux', 'nsubj:outer', 'obl:agent', 'expl:impers',
                 'expl', 'discourse', 'orphan', 'dislocated', 'flat', 'flat:foreign',
                 'iobj', 'vocative', 'csubj:outer', 'list', 'reparandum', 'csubj:pass']

DEPREL_LABELS_TO_IDX = {i: idx for idx, i in enumerate(DEPREL_LABELS)}
IDX_TO_DEPREL_LABELS = {i: j for j, i in DEPREL_LABELS_TO_IDX.items()}

# ── Para trocar de modelo, altere apenas estas variáveis ──────────────────────
# mBERT           : MODEL_NAME = 'google-bert/bert-base-multilingual-cased'  FOLD = ?
# BERTimbau-Base  : MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'      FOLD = ?
# BERTimbau-Large : MODEL_NAME = 'neuralmind/bert-large-portuguese-cased'     FOLD = ?
# ModernJabutica  : MODEL_NAME = 'amadeusai/modernJabuticaBERT-Base-1k'      FOLD = ?
# (consulte ablacao_biaffine_results.jsonl para identificar o melhor fold por LAS)
MODEL_NAME = 'amadeusai/modernJabuticaBERT-Base-1k'
FOLD       = 0  # Melhor fold por LAS de validação

FINETUNED_MODEL_PATH = f'./best_models_ablacao_biaffine/{MODEL_NAME.replace("/", "_")}'
TOKENIZER_NAME       = MODEL_NAME
# ──────────────────────────────────────────────────────────────────────────────

In [4]:
# ============================================================
# Arquitetura Biaffine — ablação sem UPOS
# ============================================================
import torch
import torch.nn as nn
import numpy as np
from typing import Optional

from transformers import BertModel, BertPreTrainedModel


# ─────────────────────────────────────────────
# 1.  MLP não-linear com inicialização segura
# ─────────────────────────────────────────────
class MLP(nn.Module):
    """Projeção não-linear usada antes das camadas biaffine."""

    def __init__(self, in_features: int, out_features: int, dropout: float = 0.33):
        super().__init__()
        self.linear     = nn.Linear(in_features, out_features)
        self.activation = nn.ELU()
        self.norm       = nn.LayerNorm(out_features)
        self.dropout    = nn.Dropout(dropout)
        nn.init.orthogonal_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.norm(self.activation(self.linear(x))))


# ─────────────────────────────────────────────
# 2.  Biaffine  (Dozat & Manning, 2017)
# ─────────────────────────────────────────────
class Biaffine(nn.Module):
    def __init__(self, in_features: int, out_features: int = 1,
                 bias_x: bool = True, bias_y: bool = True):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.bias_x = bias_x
        self.bias_y = bias_y
        self.weight = nn.Parameter(torch.zeros(
            out_features,
            in_features + int(bias_x),
            in_features + int(bias_y),
        ))
        nn.init.normal_(self.weight, std=1.0 / in_features)

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        if self.bias_x:
            x = torch.cat([x, x.new_ones(*x.shape[:-1], 1)], dim=-1)
        if self.bias_y:
            y = torch.cat([y, y.new_ones(*y.shape[:-1], 1)], dim=-1)
        return torch.einsum('bih,ohk,bjk->boij', x, self.weight, y)

In [5]:
# ─────────────────────────────────────────────
# 3.  Modelo MTL — BERT — ablação sem UPOS
# ─────────────────────────────────────────────
class MultiTaskSentencePredictionEncoder(BertPreTrainedModel):
    """
    Parser de dependências para Universal Dependencies (ablação sem UPOS):
      • HEAD    → biaffine arc  (out=1)
      • DEPREL  → biaffine rel  (out=num_deprel), avaliado na head predita
    """
    def __init__(self, config, num_deprel_labels: int,
                 arc_hidden: int = 500, rel_hidden: int = 100, mlp_dropout: float = 0.33):
        super().__init__(config)
        self.num_deprel_labels = num_deprel_labels
        self.arc_hidden = arc_hidden
        self.rel_hidden = rel_hidden
        config.num_deprel_labels = num_deprel_labels
        config.arc_hidden = arc_hidden
        config.rel_hidden = rel_hidden

        self.bert = BertModel(config, add_pooling_layer=False)
        encoder_dropout = (
            config.classifier_dropout
            if getattr(config, "classifier_dropout", None) is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(encoder_dropout)

        self.arc_head_mlp = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.arc_dep_mlp  = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.rel_head_mlp = MLP(config.hidden_size, rel_hidden, mlp_dropout)
        self.rel_dep_mlp  = MLP(config.hidden_size, rel_hidden, mlp_dropout)

        self.arc_biaffine = Biaffine(arc_hidden, out_features=1,                bias_x=True, bias_y=False)
        self.rel_biaffine = Biaffine(rel_hidden, out_features=num_deprel_labels, bias_x=True, bias_y=True)
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, Biaffine):
            nn.init.normal_(module.weight, std=1.0 / module.in_features)
        elif isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias); nn.init.ones_(module.weight)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None,
                deprel_label=None, head_label=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        seq = self.dropout(outputs.last_hidden_state)
        B, L, _ = seq.shape

        h_arc_dep  = self.arc_dep_mlp(seq)
        h_arc_head = self.arc_head_mlp(seq)
        logits_head = self.arc_biaffine(h_arc_dep, h_arc_head).squeeze(1)

        if attention_mask is not None:
            pad_mask = (attention_mask == 0).unsqueeze(1)
            logits_head = logits_head.masked_fill(pad_mask, -1e4)

        h_rel_dep  = self.rel_dep_mlp(seq)
        h_rel_head = self.rel_head_mlp(seq)
        logits_rel = self.rel_biaffine(h_rel_dep, h_rel_head)

        arc_preds = logits_head.argmax(-1).clamp(0, L - 1)
        idx_pred  = arc_preds.unsqueeze(-1).unsqueeze(-1).expand(B, L, 1, self.num_deprel_labels)
        logits_rel_t      = logits_rel.permute(0, 2, 3, 1).contiguous()
        logits_deprel_out = logits_rel_t.gather(2, idx_pred).squeeze(2)

        loss = None
        if head_label is not None and deprel_label is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            oob = (head_label != -100) & (head_label >= L)
            head_c, dep_c = head_label.clone(), deprel_label.clone()
            head_c[oob] = -100; dep_c[oob] = -100
            loss_head = loss_fct(logits_head.reshape(B*L, L), head_c.reshape(-1))
            safe = head_c.clamp(0, L-1)
            idx_g = safe.unsqueeze(-1).unsqueeze(-1).expand(B, L, 1, self.num_deprel_labels)
            lg_t  = logits_rel.permute(0, 2, 3, 1).contiguous()
            loss_dep = loss_fct(lg_t.gather(2, idx_g).squeeze(2).reshape(B*L, self.num_deprel_labels),
                                dep_c.reshape(-1))
            loss = loss_head + loss_dep

        if loss is not None:
            return (loss, logits_deprel_out, logits_head)
        return (logits_deprel_out, logits_head)

In [6]:
from transformers import ModernBertModel, ModernBertPreTrainedModel

# ─────────────────────────────────────────────
# 3b.  Modelo MTL — ModernBERT — ablação sem UPOS
# ─────────────────────────────────────────────
class MultiTaskSentencePredictionEncoderModern(ModernBertPreTrainedModel):
    """
    Parser de dependências para Universal Dependencies (ModernBERT, sem UPOS):
      • HEAD    → biaffine arc  (out=1)
      • DEPREL  → biaffine rel  (out=num_deprel), avaliado na head predita
    """
    def __init__(self, config, num_deprel_labels: int,
                 arc_hidden: int = 500, rel_hidden: int = 100, mlp_dropout: float = 0.33):
        super().__init__(config)
        self.num_deprel_labels = num_deprel_labels
        self.arc_hidden = arc_hidden
        self.rel_hidden = rel_hidden
        config.num_deprel_labels = num_deprel_labels
        config.arc_hidden = arc_hidden
        config.rel_hidden = rel_hidden

        self.model = ModernBertModel(config)
        encoder_dropout = (
            getattr(config, "classifier_dropout", None)
            or getattr(config, "hidden_dropout_prob", None)
            or getattr(config, "embedding_dropout", 0.1)
        )
        self.dropout = nn.Dropout(encoder_dropout)

        self.arc_head_mlp = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.arc_dep_mlp  = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.rel_head_mlp = MLP(config.hidden_size, rel_hidden, mlp_dropout)
        self.rel_dep_mlp  = MLP(config.hidden_size, rel_hidden, mlp_dropout)

        self.arc_biaffine = Biaffine(arc_hidden, out_features=1,                bias_x=True, bias_y=False)
        self.rel_biaffine = Biaffine(rel_hidden, out_features=num_deprel_labels, bias_x=True, bias_y=True)
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, Biaffine):
            nn.init.normal_(module.weight, std=1.0 / module.in_features)
        elif isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias); nn.init.ones_(module.weight)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None,
                deprel_label=None, head_label=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        seq = self.dropout(outputs.last_hidden_state)
        B, L, _ = seq.shape

        h_arc_dep  = self.arc_dep_mlp(seq)
        h_arc_head = self.arc_head_mlp(seq)
        logits_head = self.arc_biaffine(h_arc_dep, h_arc_head).squeeze(1)

        if attention_mask is not None:
            pad_mask = (attention_mask == 0).unsqueeze(1)
            logits_head = logits_head.masked_fill(pad_mask, -1e4)

        h_rel_dep  = self.rel_dep_mlp(seq)
        h_rel_head = self.rel_head_mlp(seq)
        logits_rel = self.rel_biaffine(h_rel_dep, h_rel_head)

        arc_preds = logits_head.argmax(-1).clamp(0, L - 1)
        idx_pred  = arc_preds.unsqueeze(-1).unsqueeze(-1).expand(B, L, 1, self.num_deprel_labels)
        logits_rel_t      = logits_rel.permute(0, 2, 3, 1).contiguous()
        logits_deprel_out = logits_rel_t.gather(2, idx_pred).squeeze(2)

        loss = None
        if head_label is not None and deprel_label is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            oob = (head_label != -100) & (head_label >= L)
            head_c, dep_c = head_label.clone(), deprel_label.clone()
            head_c[oob] = -100; dep_c[oob] = -100
            loss_head = loss_fct(logits_head.reshape(B*L, L), head_c.reshape(-1))
            safe = head_c.clamp(0, L-1)
            idx_g = safe.unsqueeze(-1).unsqueeze(-1).expand(B, L, 1, self.num_deprel_labels)
            lg_t  = logits_rel.permute(0, 2, 3, 1).contiguous()
            loss_dep = loss_fct(lg_t.gather(2, idx_g).squeeze(2).reshape(B*L, self.num_deprel_labels),
                                dep_c.reshape(-1))
            loss = loss_head + loss_dep

        if loss is not None:
            return (loss, logits_deprel_out, logits_head)
        return (logits_deprel_out, logits_head)

In [7]:
# ─────────────────────────────────────────────
# 4.  Fábrica de modelos — ablação sem UPOS
# ─────────────────────────────────────────────
def build_model(
    name_model:        str,
    num_deprel_labels: int,
    arc_hidden:  int   = 500,
    rel_hidden:  int   = 100,
    mlp_dropout: float = 0.33,
):
    from transformers import AutoConfig
    config = AutoConfig.from_pretrained(name_model)
    if config.model_type == "modernbert":
        cls = MultiTaskSentencePredictionEncoderModern
    else:
        cls = MultiTaskSentencePredictionEncoder
    print(f"[build_model] model_type='{config.model_type}' → {cls.__name__}")
    return cls.from_pretrained(
        name_model,
        config=config,
        num_deprel_labels=num_deprel_labels,
        arc_hidden=arc_hidden,
        rel_hidden=rel_hidden,
        mlp_dropout=mlp_dropout,
        _fast_init=False,
    ).to("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
from transformers import AutoConfig

# Config sempre carregado do checkpoint: garante arquitetura correta
MODEL_CONFIG = AutoConfig.from_pretrained(FINETUNED_MODEL_PATH)
TOKENIZER    = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

print(f'Modelo        : {MODEL_NAME}  |  Fold: {FOLD}')
print(f'Arquitetura   : {MODEL_CONFIG.model_type} | hidden_size: {MODEL_CONFIG.hidden_size}')
print(f'Checkpoint    : {FINETUNED_MODEL_PATH}')
print(f'Tokenizer     : {TOKENIZER_NAME}')

Modelo        : amadeusai/modernJabuticaBERT-Base-1k  |  Fold: 0
Arquitetura   : modernbert | hidden_size: 768
Checkpoint    : ./best_models_ablacao_biaffine/amadeusai_modernJabuticaBERT-Base-1k
Tokenizer     : amadeusai/modernJabuticaBERT-Base-1k


In [9]:
# Carregando modelo biaffine de ablação (sem UPOS)
model = build_model(
    FINETUNED_MODEL_PATH,
    num_deprel_labels=len(DEPREL_LABELS),
)
print('Model loaded successfully...')

[build_model] model_type='modernbert' → MultiTaskSentencePredictionEncoderModern


Loading weights: 100%|██████████| 152/152 [00:00<00:00, 12580.10it/s]


Model loaded successfully...


In [10]:
print(len(DEPREL_LABELS))

44


In [11]:
from transformers import AutoConfig

In [12]:
import pandas as pd
import torch
from tqdm import tqdm

def get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):
    predictions_deprel = []
    predictions_head   = []

    probability_deprel = []
    probability_head   = []

    model.eval()
    model.to(device)

    for tokens in tqdm(sentences):
        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            model_outputs = model(**inputs)

        logits_deprel = model_outputs[0]  # cabeça 0: DEPREL (na head predita)
        logits_head   = model_outputs[1]  # cabeça 1: HEAD

        word_ids = inputs.word_ids(batch_index=0)

        sent_preds_deprel, sent_preds_head = [], []
        sent_probs_deprel, sent_probs_head = [], []

        for token_idx in range(len(tokens)):
            subtoken_idxs = [i for i, w_id in enumerate(word_ids) if w_id == token_idx]

            if subtoken_idxs:
                first_sub = subtoken_idxs[0]
                prob_deprel = torch.softmax(logits_deprel[0, first_sub], dim=-1)
                prob_head   = torch.softmax(logits_head[0,   first_sub], dim=-1)

                pred_deprel = torch.argmax(prob_deprel).item()
                pred_head   = torch.argmax(prob_head).item()

                sent_preds_deprel.append(IDX_TO_DEPREL_LABELS[pred_deprel])
                sent_preds_head.append(pred_head)

                sent_probs_deprel.append(prob_deprel[pred_deprel].item())
                sent_probs_head.append(prob_head[pred_head].item())
            else:
                sent_preds_deprel.append(None)
                sent_preds_head.append(None)
                sent_probs_deprel.append(None)
                sent_probs_head.append(None)

        predictions_deprel.append(sent_preds_deprel)
        predictions_head.append(sent_preds_head)
        probability_deprel.append(sent_probs_deprel)
        probability_head.append(sent_probs_head)

    return pd.DataFrame({
        "tokens":                  sentences,
        "deprel_predictions":      predictions_deprel,
        "deprel_pred_probability": probability_deprel,
        "head_predictions":        predictions_head,
        "head_pred_probability":   probability_head,
    })

In [13]:
from datasets import load_from_disk

dataset = load_from_disk('/home/guilhermelima/msc/data_dois/complaints_dataset_obj_outxpos')
test_dataset = dataset['test']
print(f"Test set: {len(test_dataset)} sentenças")

Test set: 1683 sentenças


In [14]:
test_sentences = test_dataset['tokens']
test_deprel    = test_dataset['deprel']
test_head      = test_dataset['head_tags']

print(f"Sentenças: {len(test_sentences)}")
print(f"Exemplo tokens : {test_sentences[0]}")
print(f"Exemplo deprel : {test_deprel[0]}")
print(f"Exemplo heads  : {test_head[0]}")

Sentenças: 1683
Exemplo tokens : ['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']
Exemplo deprel : ['det', 'nsubj', 'flat:name', 'advmod', 'root', 'det', 'obj', 'punct']
Exemplo heads  : [2, 5, 2, 5, 0, 7, 5, 5]


In [15]:
# test_sentences, test_deprel, test_head já definidos na célula anterior
print(f"Total de sentenças de teste: {len(test_sentences)}")
print(f"Primeira sentença: {test_sentences[0]}")

Total de sentenças de teste: 1683
Primeira sentença: ['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']


In [16]:
test_sentences

Column([['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.'], ['A', 'Odebrecht', 'pagou', '300', '%', 'a', 'mais', 'por', 'o', 'direito', 'de', 'explorar', 'o', 'aeroporto', 'de', 'o', 'Galeão', '.'], ['Em', 'o', 'começo', 'de', 'o', 'século', ',', 'a', 'JBS/Friboi', 'chegava', 'a', 'o', 'grupo', 'de', 'as', '400', 'maiores', '.'], ['Os', 'sons', 'indesejáveis', 'emitidos', 'por', 'uma', 'porta', ',', 'por', 'exemplo', ',', 'são', 'eliminados', 'por', 'R$', '150', '.'], ['Que', 'foi', 'herança', 'de', 'o', 'PT', ',', 'que', 'nos', 'deixou', 'esse', 'rombo', ',', 'disse', 'Doria', '.'], ...])

In [17]:
model

MultiTaskSentencePredictionEncoderModern(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
  

In [18]:
predict_test_df = get_predictions_on_dataframe(test_sentences, model, TOKENIZER)

100%|██████████| 1683/1683 [01:02<00:00, 27.01it/s]


In [19]:
predict_test_df

,tokens,deprel_predictions,deprel_pred_probability,head_predictions,head_pred_probability
0,"[O, Capitão, América, também, bajulou, o, tuca...","[det, nsubj, flat:name, advmod, root, det, obj...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9999998807907...","[2, 5, 2, 5, 0, 7, 5, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]"
1,"[A, Odebrecht, pagou, 300, %, a, mais, por, o,...","[det, nsubj, root, nummod, obj, case, advmod, ...","[1.0, 1.0, 1.0, 0.9999998807907104, 0.99999988...","[2, 3, 0, 5, 3, 7, 5, 10, 10, 3, 12, 10, 14, 1...","[1.0, 1.0, 1.0, 0.999283492565155, 1.0, 1.0, 0..."
2,"[Em, o, começo, de, o, século, ,, a, JBS/Fribo...","[case, det, obl, case, det, nmod, punct, det, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.999...","[3, 3, 12, 6, 6, 3, 3, 9, 12, 0, 15, 15, 12, 1...","[1.0, 1.0, 0.9996883869171143, 1.0, 1.0, 1.0, ..."
3,"[Os, sons, indesejáveis, emitidos, por, uma, p...","[det, nsubj:pass, amod, acl, case, det, obl:ag...","[1.0, 0.9999785423278809, 0.9999998807907104, ...","[2, 13, 2, 2, 7, 7, 4, 10, 10, 13, 10, 13, 0, ...","[1.0, 1.0, 1.0, 1.0, 0.9999998807907104, 1.0, ..."
4,"[Que, foi, herança, de, o, PT, ,, que, nos, de...","[nsubj, cop, ccomp, case, det, nmod, punct, ns...","[0.9999972581863403, 1.0, 0.9999090433120728, ...","[3, 3, 14, 6, 6, 3, 10, 10, 10, 6, 12, 10, 10,...","[1.0, 1.0, 0.9999995231628418, 1.0, 1.0, 1.0, ..."
...,...,...,...,...,...
1678,"[Julia, Louis-Dreyfus, ganhou, por, a, sexta, ...","[nsubj, flat:name, root, case, det, amod, obl,...","[0.9999997615814209, 0.9999982118606567, 1.0, ...","[3, 1, 0, 7, 7, 7, 3, 7, 10, 3, 13, 13, 3, 16,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.99999785..."
1679,"[Até, a, família, julga, mais, e, apoia, menos...","[advmod, det, nsubj, ccomp:speech, advmod, cc,...","[0.9999587535858154, 0.9999996423721313, 0.999...","[3, 3, 4, 15, 4, 7, 4, 7, 13, 11, 13, 13, 7, 4...","[0.9999431371688843, 1.0, 1.0, 0.9999995231628..."
1680,"[Mas, há, episódios, que, indicam, em, Damião,...","[cc, root, obj, nsubj, acl:relcl, case, obl, d...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.999...","[2, 0, 2, 5, 3, 7, 5, 9, 5, 9, 2]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
1681,"["", Mas, Che, permanece, puro, ,, de, certo, m...","[punct, cc, nsubj, ccomp:speech, xcomp, punct,...","[1.0, 0.9999997615814209, 1.0, 0.9999995231628...","[4, 4, 4, 13, 4, 9, 9, 9, 4, 4, 4, 13, 0, 13]","[1.0, 1.0, 1.0, 1.0, 1.0, 0.9999998807907104, ..."


In [20]:
for i in range(1):
    print(
        f"Exemplo {i} - Tokens: {len(predict_test_df['tokens'][i])} | "
        f"DEPREL: {len(predict_test_df['deprel_predictions'][i])} | "
        f"HEAD: {len(predict_test_df['head_predictions'][i])}"
    )

Exemplo 0 - Tokens: 8 | DEPREL: 8 | HEAD: 8


In [21]:
print(test_sentences[0])

['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']


In [22]:
def compute_dependency_metrics(test_sentences, test_deprel, test_head, predict_df):
    """
    Métricas para análise de dependências — ablação sem UPOS:
      UAS — Unlabeled Attachment Score: HEAD correto
      LAS — Labeled Attachment Score:   HEAD + DEPREL corretos
    """
    total       = 0
    uas_correct = 0
    las_correct = 0
    skipped     = 0

    for i in range(len(test_sentences)):
        gold_deprel = test_deprel[i]
        gold_head   = test_head[i]

        pred_deprel = predict_df['deprel_predictions'].iloc[i]
        pred_head   = predict_df['head_predictions'].iloc[i]

        for j in range(len(gold_head)):
            if pred_head[j] is None or pred_deprel[j] is None:
                skipped += 1
                continue

            total += 1

            # UAS: HEAD correto
            if pred_head[j] == gold_head[j]:
                uas_correct += 1
                # LAS: HEAD correto E DEPREL correto
                if pred_deprel[j] == gold_deprel[j]:
                    las_correct += 1

    return {
        'uas':          uas_correct / total if total > 0 else 0,
        'las':          las_correct / total if total > 0 else 0,
        'total_tokens': total,
        'skipped':      skipped,
    }

In [23]:
metrics = compute_dependency_metrics(
    test_sentences, test_deprel, test_head, predict_test_df
)

print(f"UAS           : {metrics['uas']:.4f}")
print(f"LAS           : {metrics['las']:.4f}")
print(f"Total tokens  : {metrics['total_tokens']}")
print(f"Ignorados     : {metrics['skipped']}")

UAS           : 0.9004
LAS           : 0.8845
Total tokens  : 33580
Ignorados     : 0


In [24]:
#text = [["O", "gato", "preto", "dorme", "no", "sofá", "."]]
#text = [["A", "menina", "brinca", "no", "parque", "."]]
#text = [["Se", "chover", ",", "o", "jogo", "será", "cancelado", "."]]
text = [['Mas', 'por', 'não', 'existir', 'um', 'marco', 'legal', 'há', 'uma',
         'insegurança', 'por', 'parte', 'dos', 'investidores', '"', ',', 'destacou', '.']]

In [25]:
retorno = get_predictions_on_dataframe(text, model, TOKENIZER)

100%|██████████| 1/1 [00:00<00:00, 27.39it/s]


In [26]:
retorno

,tokens,deprel_predictions,deprel_pred_probability,head_predictions,head_pred_probability
0,"[Mas, por, não, existir, um, marco, legal, há,...","[cc, mark, advmod, advcl, det, nsubj, amod, ad...","[1.0, 0.9999953508377075, 1.0, 1.0, 0.99999988...","[17, 4, 4, 8, 6, 8, 6, 4, 10, 8, 12, 8, 14, 12...","[0.9998216032981873, 1.0, 1.0, 0.9981573224067..."


In [27]:
print('Gold deprel sentença 2:', test_deprel[2])
print('Pred deprel sentença 2:', predict_test_df['deprel_predictions'].iloc[2])

Gold deprel sentença 2: ['case', 'det', 'obl', 'case', 'det', 'nmod', 'punct', 'det', 'nsubj', 'root', 'case', 'det', 'obl', 'case', 'det', 'nmod', 'amod', 'punct']
Pred deprel sentença 2: ['case', 'det', 'obl', 'case', 'det', 'nmod', 'punct', 'det', 'nsubj', 'root', 'case', 'det', 'obl', 'case', 'det', 'nmod', 'amod', 'punct']


In [28]:
out_csv = f'./predict_test_ablacao_biaffine_{MODEL_NAME.replace("/", "_")}_fold{FOLD}.csv'
predict_test_df.to_csv(out_csv, index=False)
print(f"Salvo em: {out_csv}")

Salvo em: ./predict_test_ablacao_biaffine_amadeusai_modernJabuticaBERT-Base-1k_fold0.csv
